# 06 — Data Preprocessing & Model Building

This notebook implements and verifies the tasks from **Epic 3: Data Pre-processing** and **Epic 4: Model Building**:
1. **Handling Outliers** using IQR winsorization (capping).
2. **Handling Categorical Values** using one-hot encoding.
3. **Splitting Data into Training and Test Sets** with stratified splitting.
4. **Feature Scaling** using StandardScaler.
5. **Decision Tree Model** training and hyperparameter tuning.
6. **Random Forest Model** training and hyperparameter tuning.

In [ ]:
import sys
import os
import pandas as pd
import numpy as np

# Add source code directory to path
sys.path.append(os.path.abspath(r"../../5. Project Development Phase/src"))

from preprocessing import load_data, handle_outliers, handle_missing_values, encode_categories, split_data, scale_features
from decision_tree import train_decision_tree, tune_hyperparameters as tune_dt
from random_forest import train_random_forest, tune_hyperparameters as tune_rf

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report

# Load dataset
filepath = r"../../3. Project Design Phase/dataset/raw/flood dataset.xlsx"
df = load_data(filepath)
df = handle_missing_values(df)
print(f"Loaded dataset with shape: {df.shape}")
df.head()

: 

## 1. Handling Outliers

We use **IQR Winsorization (capping)** to handle outliers. Capping outliers preserves all rows (unlike dropping outliers), which is critical since our dataset is small (115 rows) and has a highly imbalanced target column (`flood` has only 16 positive examples).

In [ ]:
# Inspect statistics before outlier capping
print("Max values BEFORE capping outliers:")
print(df.max())

# Apply outlier treatment
df_clean = handle_outliers(df)

print("\nMax values AFTER capping outliers:")
print(df_clean.max())

## 2. Handling Categorical Values

Even though all columns in this dataset are currently numeric, we run `encode_categories` to ensure robust handling of non-numeric data types (e.g. converting categorical object columns to one-hot representation).

In [ ]:
# Encode categories (if any)
df_encoded = encode_categories(df_clean)
print(f"Encoded dataset shape: {df_encoded.shape}")
print(df_encoded.dtypes)

## 3. Splitting Data into Training and Test Sets

We separate features and target, and split into train and test sets using **stratified splitting** (`stratify=y`) to maintain the target class distribution in both splits.

In [ ]:
X = df_encoded.drop(columns=['flood'])
y = df_encoded['flood']

X_train, X_test, y_train, y_test = split_data(X, y, test_size=0.2, random_state=42)

print(f"Train size: {X_train.shape[0]} samples")
print(f"Test size:  {X_test.shape[0]} samples")
print("\nTarget distribution in Training Set:")
print(y_train.value_counts(normalize=True))
print("\nTarget distribution in Test Set:")
print(y_test.value_counts(normalize=True))

## 4. Feature Scaling

We use `StandardScaler` to bring features onto the same scale (mean=0, variance=1) which is critical for distance-based and regularized algorithms.

In [ ]:
X_train_scaled, X_test_scaled, scaler = scale_features(X_train, X_test, scaler_type="standard")
print("Training Set summary statistics after scaling:")
X_train_scaled.describe().loc[['mean', 'std', 'min', 'max']]

## 5. Decision Tree Model

We train a `DecisionTreeClassifier` and tune its hyper-parameters using grid search over depth, min samples split, and leaf nodes, using `f1` as the grid scoring metric.

In [ ]:
dt_grid = {
    'max_depth': [3, 5, 8, 10, None],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

best_dt, best_dt_params = tune_dt(X_train_scaled, y_train, dt_grid)
print(f"Best Decision Tree parameters found: {best_dt_params}")

# Predict & Evaluate
dt_preds = best_dt.predict(X_test_scaled)
print("\nDecision Tree Test Report:")
print(classification_report(y_test, dt_preds, zero_division=0))

## 6. Random Forest Model

We train a `RandomForestClassifier` with balanced class weights to penalize minority class errors and search over trees and maximum depth.

In [ ]:
rf_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 8, None],
    'min_samples_split': [2, 5, 10]
}

best_rf, best_rf_params = tune_rf(X_train_scaled, y_train, rf_grid)
print(f"Best Random Forest parameters found: {best_rf_params}")

# Predict & Evaluate
rf_preds = best_rf.predict(X_test_scaled)
print("\nRandom Forest Test Report:")
print(classification_report(y_test, rf_preds, zero_division=0))

## Summary of Findings

### Data Analysis Key Findings
- The target variable is highly imbalanced with **16 flood cases** and **99 non-flood cases** out of 115 samples.
- Outliers capping successfully adjusted the maximum rainfall and temperature ranges without losing valuable flood rows.
- Scaling has centered features around 0, making model weights comparable.
- GridSearchCV hyperparameter tuning has optimized Decision Tree and Random Forest parameters based on minority class F1-Score.

### Insights or Next Steps
- Run all cells in this notebook to see output statistics and compare model performance metrics on your local environment.